In [ ]:
import os
import numpy as np
from design import Design
from generators import FourStupid, HacklGenerator
from geometry import analyze_results, plot_barriers

project_name = "SynRM_test"
design_name = "Design01"
path_data = os.path.join(os.getcwd(), 'data')
path_results = 'results'
for path in [path_data, path_results]:
    os.makedirs(path, exist_ok=True)
file_name_aedt = f'{path_data}/{project_name}.aedt'
plot_design = True
n_designs = 10

# Define constants
AEDT_VERSION = "2024.1"
NUM_CORES = 4
NG_MODE = True  #non-graphical mode
CLS_EXIT = True #close on exit

if not os.path.exists(file_name_aedt):
    design = Design.create(
        project_name, design_name, file_name_aedt,
        version=AEDT_VERSION,
        non_graphical=NG_MODE,
        new_desktop=False,
        close_on_exit=CLS_EXIT,
    )
else:
    design = Design.load(
        file_name_aedt,
        version=AEDT_VERSION,
        non_graphical=NG_MODE,
        new_desktop=False,
        close_on_exit=CLS_EXIT,
    )

In [ ]:
r_stator_end = 0.7
offset = 0.7 / 2
generator_stupid = FourStupid(design, r_stator_end, offset=offset)
generator_hackl = HacklGenerator(design, r_stator_end, offset=offset)
generators = [generator_stupid, generator_hackl]

for i in range(0, n_designs):
    for generator in generators:
        barriers1 = generator.random_barriers()
        barriers2 = generator.split_barriers(barriers1)
        for j, barriers in enumerate([barriers1, barriers2]):
            design.add_rotor()

            for barrier in barriers:
                design.add_rotor_barrier(barrier)

            # Compute the torque
            Tor = design.compute(NUM_CORES)
            TorAvg, _, TorRippleRms = analyze_results(Tor)

            # Delete the rotor
            design.delete_rotor()

            # Potentially save the design
            if plot_design:
                title = f'Torque mean value: {np.round(TorAvg,2)} Nm, ripple relative value: {np.round(TorRippleRms,2)} %'
                file_name = f'{path_results}/design_{generator.name}_{i}_{j}'
                plot_barriers(barriers, design, title=title, file_name = f"{file_name}.png")
                generator.save_barriers(f"{file_name}.npz")

In [ ]:
design.close_project()